# EDA do WebSirenes

Analise visual dos artefatos gerados pela auditoria versionada do WebSirenes. Este notebook nao define regras de qualidade nem altera a fonte observacional.

## Pre-requisito

Execute `nowcasting-audit-websirene` antes de abrir este notebook. Os arquivos em `outputs/analysis/websirene/qc/v1` sao a fonte unica das classificacoes `accepted`, `suspect` e `rejected`.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from nowcasting.websirene_qc import repair_mojibake


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DEFAULT_AUDIT_DIR = PROJECT_ROOT / 'outputs' / 'analysis' / 'websirene' / 'qc' / 'v1'
AUDIT_DIR = Path(os.environ.get('WEBSIRENE_AUDIT_DIR', DEFAULT_AUDIT_DIR))

required_files = ('summary.json', 'station_summary.csv', 'year_summary.csv', 'station_whitelist.csv')
missing = [name for name in required_files if not (AUDIT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'Artefatos de auditoria ausentes em {AUDIT_DIR}: {missing}. ' 
        'Execute nowcasting-audit-websirene antes de abrir este notebook.'
    )

print(f'Projeto: {PROJECT_ROOT}')
print(f'Auditoria: {AUDIT_DIR}')

In [ ]:
with (AUDIT_DIR / 'summary.json').open(encoding='utf-8') as handle:
    audit_summary = json.load(handle)

station_summary = pd.read_csv(AUDIT_DIR / 'station_summary.csv')
station_summary['station_name'] = station_summary['station_name'].map(repair_mojibake)
year_summary = pd.read_csv(AUDIT_DIR / 'year_summary.csv')
station_whitelist = pd.read_csv(AUDIT_DIR / 'station_whitelist.csv')
station_whitelist['station_name'] = station_whitelist['station_name'].map(repair_mojibake)
station_whitelist['accepted_pct'] = 100 * station_whitelist['accepted_fraction']

station_summary['accepted_pct'] = 100 * station_summary['accepted_fraction']
station_summary['suspect_pct'] = 100 * station_summary['suspect_fraction']
station_summary['rejected_pct'] = 100 * station_summary['rejected_fraction']

print('Versao de QC:', audit_summary['qc_version'])
print('Arquivos auditados:', audit_summary['files_processed'])
print('Estacoes auditadas:', audit_summary['stations_audited'])
print('Estacoes aprovadas:', audit_summary['stations_approved'])

## Resumo global de qualidade

In [ ]:
global_metrics = pd.Series(audit_summary['observation_status_counts'], name='observacoes').to_frame()
global_metrics['percentual'] = 100 * global_metrics['observacoes'] / global_metrics['observacoes'].sum()
display(global_metrics)

ax = global_metrics['observacoes'].plot.pie(autopct='%.2f%%', ylabel='', figsize=(5, 5))
ax.set_title('Classificacao de observacoes WebSirenes')
plt.tight_layout()

## Cobertura temporal

In [ ]:
coverage_by_year = (
    year_summary[year_summary['accepted'] > 0]
    .groupby('year')['station_id']
    .nunique()
    .rename('estacoes_com_observacoes_aceitas')
    .reset_index()
)

ax = coverage_by_year.plot(x='year', y='estacoes_com_observacoes_aceitas', kind='bar', legend=False, figsize=(12, 4))
ax.set(xlabel='Ano', ylabel='Estacoes com observacoes aceitas', title='Cobertura anual apos QC')
plt.tight_layout()
display(coverage_by_year)

In [ ]:
required_years = set(range(2012, 2025))
years_by_station = (
    year_summary[year_summary['accepted'] > 0]
    .groupby('station_id')['year']
    .agg(lambda years: sorted(set(years)))
    .rename('anos_com_observacoes_aceitas')
    .reset_index()
)
coverage_table = station_summary[['station_id', 'station_name', 'station_status', 'first_timestamp', 'last_timestamp']].merge(
    years_by_station, on='station_id', how='left'
)
coverage_table['cobre_2012_2024'] = coverage_table['anos_com_observacoes_aceitas'].map(
    lambda years: required_years.issubset(set(years)) if isinstance(years, list) else False
)
print('Estacoes com observacoes aceitas em todos os anos de 2012-2024:', int(coverage_table['cobre_2012_2024'].sum()))
display(coverage_table.sort_values(['station_status', 'station_id']))

## Aprovacao e priorizacao de estacoes

In [ ]:
station_columns = [
    'station_id', 'station_name', 'station_status', 'station_status_reasons',
    'observations', 'accepted_observations', 'accepted_pct',
    'suspect_observations', 'suspect_pct', 'rejected_observations', 'rejected_pct',
    'years_with_accepted_data', 'max_m15', 'flag_counts',
]
station_table = station_summary[station_columns].sort_values(
    ['station_status', 'rejected_observations', 'suspect_observations'],
    ascending=[True, False, False],
)
display(station_table)

ax = station_table.sort_values('rejected_observations', ascending=False).head(15).plot.barh(
    x='station_name', y=['suspect_observations', 'rejected_observations'], stacked=True, figsize=(10, 6)
)
ax.set(xlabel='Observacoes', ylabel='Estacao', title='15 estacoes com mais observacoes suspeitas ou rejeitadas')
plt.tight_layout()

In [ ]:
print('Whitelist preliminar:', len(station_whitelist), 'estacoes')
display(station_whitelist[['station_id', 'station_name', 'accepted_pct', 'years_with_accepted_data', 'max_m15']])

## Inspecao de registros sinalizados

`flagged_observations.csv` contem somente linhas `suspect` ou `rejected` e pode ser usado para inspecao pontual. A classificacao oficial continua definida no script e no arquivo de configuracao da auditoria.

In [ ]:
flagged_path = AUDIT_DIR / 'flagged_observations.csv'
if flagged_path.is_file():
    flagged = pd.read_csv(flagged_path, nrows=20)
    display(flagged)
else:
    print('Nenhum arquivo de observacoes sinalizadas foi produzido pela auditoria.')

## Limites e proximos passos

- Consulte `docs/CONTROLE_QUALIDADE_WEBSIRENE.md` para as regras, limites e criterios da whitelist.
- Use `notebooks/02_geospatial/02_mapa_redes_pluviometricas.ipynb` para a distribuicao espacial conjunta de WebSirenes e AlertaRio.
- A construcao de targets e qualquer alteracao das regras de QC devem ocorrer em comandos versionados do pipeline, nunca neste notebook.